# 5.6 上下文并行深挖 (Ulysses / Ring Attention)

> 🕐 预估学习时间：40分钟

序列并行/上下文并行把超长序列切到多卡：DeepSpeed Ulysses（all-to-all 头维交换）与 Ring Attention（块状 KV 环形传递）是两条主路。

深挖点：
- 激活切分维度对比
- Ulysses all-to-all 通信量
- Ring Attention 正确性模拟
- 与 TP/PP/CP 三维组合


## 1. 为什么需要上下文并行？

TP 切隐藏维，PP 切层，都救不了 **激活 ∝ batch×seq×hidden** 在超长 seq 上的爆炸。CP 沿序列维切分。


In [ ]:
import torch
import torch.nn.functional as F
import math

torch.manual_seed(0)


def activation_bytes(batch, seq, hidden, dtype_bytes=2, layers=1):
    # rough: activations per layer ~ 2 * B * S * H (residual+mlp peak simplified)
    return batch * seq * hidden * dtype_bytes * 2 * layers


print('=== Activation Memory vs Seq ===')
for s in [4_096, 32_768, 128_000, 1_000_000]:
    single = activation_bytes(1, s, 4096, layers=4) / (1024**3)
    cp8 = activation_bytes(1, s // 8, 4096, layers=4) / (1024**3)
    print(f'seq={s:>8}: single~{single:.2f}GB  CP=8 ~{cp8:.2f}GB/rank')
print('Key: CP turns sequence-length memory into a near-linear scale-out knob.')


## 2. Ulysses：序列分片 ↔ 头分片 all-to-all

1. 每卡持有 seq/P 段、全部头  
2. all-to-all 后变为持有全部 seq、头/P  
3. 本地算注意力  
4. all-to-all 换回  

通信量与 P 相关但避免 Ring 的多步延迟积累。


In [ ]:
def ulysses_comm_bytes(batch, seq, heads, d_head, P, dtype=2):
    # two all-to-alls of Q/K/V-sized (approx 3) + output (1) => ~4 transfers of B*S*H
    H = heads * d_head
    per = batch * seq * H * dtype
    # each all-to-all moves (P-1)/P of local shard volume from each rank; total network ~ per * (P-1)/P * 2 * 2
    return 2 * 2 * per * (P - 1) / P


print('=== Ulysses Comm Volume ===')
for P in [2, 4, 8]:
    b = ulysses_comm_bytes(1, 65536, 32, 128, P)
    print(f'P={P}: ~{b/1024**3:.2f} GB total network (toy accounting)')
print('Key: Ulysses prefers fat all-to-all links inside a node/NVLink domain.')


## 3. Ring Attention：KV 块绕环传递

每卡固定拥有 Q 的一段；KV 块在环上轮转，局部累加 online softmax 统计量，数学上等价全注意力。


In [ ]:
def ring_attention(Q, K, V, ranks=4):
    '''Q,K,V: (B,H,S,D) split along S into ranks chunks; compute exact attn.'''
    B, H, S, D = Q.shape
    assert S % ranks == 0
    chunk = S // ranks
    outs = []
    for r in range(ranks):
        qs = Q[:, :, r * chunk:(r + 1) * chunk]
        # online softmax over circulating KV
        m = torch.full((B, H, chunk, 1), -1e9)
        l = torch.zeros(B, H, chunk, 1)
        o = torch.zeros(B, H, chunk, D)
        for step in range(ranks):
            src = (r + step) % ranks
            ks = K[:, :, src * chunk:(src + 1) * chunk]
            vs = V[:, :, src * chunk:(src + 1) * chunk]
            s = qs @ ks.transpose(-1, -2) / math.sqrt(D)
            m2 = torch.maximum(m, s.max(-1, keepdim=True).values)
            alpha = torch.exp(m - m2)
            p = torch.exp(s - m2)
            l = l * alpha + p.sum(-1, keepdim=True)
            o = o * alpha + p @ vs
            m = m2
        outs.append(o / l)
    return torch.cat(outs, dim=2)


B, H, S, D = 1, 2, 64, 16
Q = torch.randn(B, H, S, D)
K = torch.randn(B, H, S, D)
V = torch.randn(B, H, S, D)
ref = torch.softmax(Q @ K.transpose(-1, -2) / math.sqrt(D), -1) @ V
ring = ring_attention(Q, K, V, ranks=4)
err = (ref - ring).abs().max().item()
print('=== Ring Attention Equivalence ===')
print(f'max err={err:.2e}')
print('Key: Ring Attention is exact; latency grows with ring steps (~P).')


## 4. 组合策略

| 维度 | 切什么 | 适合 |
|------|-------|------|
| TP | 头/MLP 宽 | 节点内 |
| PP | 层 | 节点间流水 |
| CP/Ulysses | 序列 | 超长上下文 |
| DP/FSDP | 数据/参数状态 | 吞吐 |

经验：长上下文预训练常 `TP × CP` 同节点，`PP` 跨节点；推理可用环形 KV 传输或状态缓存。


In [ ]:
def pick_parallel(seq_len, node_gpus=8):
    if seq_len <= 8192:
        return {'TP': min(8, node_gpus), 'CP': 1, 'PP': 1}
    if seq_len <= 65536:
        return {'TP': 4, 'CP': 2, 'PP': 1}
    return {'TP': 2, 'CP': 4, 'PP': 2}


print('=== Heuristic Parallel Plans ===')
for s in [4096, 32768, 128000]:
    print(s, pick_parallel(s))
print('Key: Raise CP before PP when activation memory is the binding constraint.')


## 课后思考题

1. Ulysses 与 Ring 在跨节点 InfiniBand 上谁更怕延迟？
2. Causal mask 在 ring 步进中如何正确处理？
3. CP 与梯度检查点同时开时，通信-重计算如何重叠？
4. 推理期 KV 已分片时，CP 训练出的模型如何服务？

---
> 本节是上下文并行的垂直深挖。建议对照真实训练日志/线上指标复现关键实验，而不是只跑通玩具代码。
